# Abstract: The Persistence of Momentum
The momentum effect—the empirical observation that relative performance tends to persist—represents a systematic challenge to the Efficient Market Hypothesis. Since Jegadeesh and Titman (1993), this phenomenon has been established as a universal market feature rather than an anomaly.

Theoretical Framework:

- Empirical Robustness: Cross-asset and cross-jurisdictional replication confirm that price trends are statistically persistent.
- Institutional Exploitation: The continued deployment of momentum strategies by quantitative funds validates the existence of exploitable return premiums in live markets.
- Behavioral Mechanism: The momentum premium arises from delayed information diffusion and investor herding, which facilitate gradual price discovery, contradicting the theory of instantaneous market adjustment.

# First Step: Proving Market Observation
RSI snapback speed has a strong correlation with MA acceleration.
- If MA acceleration > 0, RSI snapback speed is significantly faster on the long side.
- If MA acceleration < 0, RSI snapback speed is significantly faster on the short side.

| Strong Bull (Force_POS) | Sideway Market | Strong Bear (Force_NEG) |
| :---: | :---: | :---: |
| ![Strong Bull](./image/img_1.png) | ![Stop Trading](./image/img_3.png) | ![Strong Short](./image/img_2.png) |


In [1]:
import sys
import os
import warnings
import itertools
import json

import numpy as np
import pandas as pd
from scipy import stats

# ── Helper: convert numpy types to native Python for JSON ──────────────────
def _to_native(obj):
    """Recursively convert numpy types to native Python types for JSON serialization."""
    if isinstance(obj, dict):
        return {k: _to_native(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [_to_native(v) for v in obj]
    elif isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return _to_native(obj.tolist())
    elif isinstance(obj, np.bool_):
        return bool(obj)
    return obj

# ── Suppress noise ──────────────────────────────────────────────────────────
warnings.filterwarnings('ignore')
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from utils.bybit import BybitConnector
from utils.indicators import get_wilders_rsi, sma

# ── Statsmodels ─────────────────────────────────────────────────────────────
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch, het_breuschpagan
from statsmodels.stats.api import het_goldfeldquandt, jarque_bera
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA

import matplotlib.pyplot as plt


# ═══════════════════════════════════════════════════════════════════════════
#  SECTION 1 — DATA COLLECTION
# ═══════════════════════════════════════════════════════════════════════════

print("=" * 72)
print("  ARIMA HYPOTHESIS TESTING — STREAK DURATION ANALYSIS")
print("=" * 72)

print("\n[1/7] Fetching BTCUSDT 1-min OHLCV data...")
client = BybitConnector(api_key="", api_secret="", testnet=False)
price = client.get_ohlcv("BTC/USDT:USDT", "1", 10000)
close = np.asarray(price['close'], dtype=float)
print(f"  ✓ Fetched {len(close)} bars")


# ═══════════════════════════════════════════════════════════════════════════
#  SECTION 2 — BUILD RSI STREAKS
# ═══════════════════════════════════════════════════════════════════════════

print("\n[2/7] Building RSI streak durations...")
diffs = np.diff(close)
rsi_14 = get_wilders_rsi(diffs, 14)
rsi_14_sma = sma(rsi_14, 14)

# Pad NaN to front to align length
align = len(rsi_14) - len(rsi_14_sma)
rsi_14_sma_padded = np.concatenate([np.full(align, np.nan), rsi_14_sma])

# Create streak blocks
df = pd.DataFrame({
    'rsi_14': rsi_14,
    'sma_14': rsi_14_sma_padded,
})
df = df.dropna().reset_index(drop=True)

df['rsi_above_sma'] = df['rsi_14'] > df['sma_14']
df['block'] = (df['rsi_above_sma'] != df['rsi_above_sma'].shift()).cumsum()

streak_durations = (
    df.groupby('block')
      .agg(duration=('rsi_above_sma', 'size'),
           is_above=('rsi_above_sma', 'first'))
      .reset_index(drop=True)
)

durations = streak_durations['duration'].values
durations_log = np.log1p(durations)  # log(1 + duration)

print(f"  ✓ Total streaks: {len(durations)}")
print(f"  ✓ Duration range: [{durations.min()}, {durations.max()}] bars")
print(f"  ✓ Duration mean:  {durations.mean():.2f} bars")
print(f"  ✓ Duration std:   {durations.std():.2f} bars")
print(f"  ✓ Skewness:       {stats.skew(durations):.4f}")
print(f"  ✓ Kurtosis:       {stats.kurtosis(durations):.4f} (excess)")

  ARIMA HYPOTHESIS TESTING — STREAK DURATION ANALYSIS

[1/7] Fetching BTCUSDT 1-min OHLCV data...
  ✓ Fetched 9999 bars

[2/7] Building RSI streak durations...
  ✓ Total streaks: 1808
  ✓ Duration range: [1, 30] bars
  ✓ Duration mean:  5.52 bars
  ✓ Duration std:   5.36 bars
  ✓ Skewness:       1.3926
  ✓ Kurtosis:       1.4587 (excess)


# Stationarity Test ADF and KPSS
Double validation using ADF to test if the series is stationary and KPSS to test if the series is non-stationary.

In [2]:
# ═══════════════════════════════════════════════════════════════════════════
#   SECTION 3 — STATIONARITY TESTS
# ═══════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 72)
print("  SECTION 3 — STATIONARITY TESTS")
print("=" * 72)

# --- 3a. Augmented Dickey-Fuller Test ---
# H0: Unit root exists (non-stationary)
# H1: Series is stationary
print("\n--- 3a. Augmented Dickey-Fuller (ADF) ---")
print("    H0: Unit root exists → series is NON-STATIONARY")
adf_result = adfuller(durations, autolag='AIC')
adf_stat, adf_pval, adf_lags, adf_nobs, adf_critvals, adf_icbest = adf_result

print(f"    Test statistic:     {adf_stat:.6f}")
print(f"    p-value:            {adf_pval:.6f}")
print(f"    Lags used:          {adf_lags}")
print(f"    Critical values:")
for key, val in adf_critvals.items():
    print(f"      {key}: {val:.6f}")
if adf_pval < 0.05:
    print(f"    → REJECT H0 (p={adf_pval:.6f} < 0.05): Series is STATIONARY")
else:
    print(f"    → FAIL to reject H0 (p={adf_pval:.6f} >= 0.05): Evidence of NON-STATIONARITY")

# --- 3b. KPSS Test ---
# H0: Series is stationary (trend-stationary or level-stationary)
# H1: Series has a unit root (non-stationary)
print("\n--- 3b. KPSS Test ---")
print("    H0: Series is STATIONARY (level or trend)")
kpss_result = kpss(durations, regression='c', nlags='auto')
kpss_stat, kpss_pval, kpss_lags, kpss_critvals = kpss_result

print(f"    Test statistic:     {kpss_stat:.6f}")
print(f"    p-value:            {kpss_pval:.6f}")
print(f"    Lags used:          {kpss_lags}")
print(f"    Critical values:")
for key, val in kpss_critvals.items():
    print(f"      {key}: {val:.6f}")
if kpss_pval < 0.05:
    print(f"    → REJECT H0 (p={kpss_pval:.6f} < 0.05): Series is NON-STATIONARY")
else:
    print(f"    → FAIL to reject H0 (p={kpss_pval:.6f} >= 0.05): Consistent with STATIONARITY")

# --- Combined stationarity verdict ---
print("\n--- Combined Stationarity Verdict ---")
adf_reject = adf_pval < 0.05
kpss_reject = kpss_pval < 0.05

print(f"  ADF  (H0=non-stationary): {'STATIONARY ✓' if adf_reject else 'NON-STATIONARY ✗'}")
print(f"  KPSS (H0=stationary):     {'NON-STATIONARY ✓' if kpss_reject else 'STATIONARY ✗'}")

if adf_reject and not kpss_reject:
    print("  ✓ Agreement: Series is STATIONARY (ADF rejects unit root, KPSS fails to reject stationarity)")
elif not adf_reject and kpss_reject:
    print("  ✓ Agreement: Series is NON-STATIONARY (ADF fails to reject unit root, KPSS rejects stationarity)")
elif adf_reject and kpss_reject:
    print("  ⚠ Conflict: ADF suggests stationarity, but KPSS suggests non-stationarity.")
    print("    → Possible structural break or trend-stationarity issues.")
else:
    print("  ⚠ Conflict: ADF suggests non-stationarity, but KPSS suggests stationarity.")
    print("    → Inconclusive results; consider inspecting the series or checking differencing.")


  SECTION 3 — STATIONARITY TESTS

--- 3a. Augmented Dickey-Fuller (ADF) ---
    H0: Unit root exists → series is NON-STATIONARY
    Test statistic:     -17.256153
    p-value:            0.000000
    Lags used:          4
    Critical values:
      1%: -3.433982
      5%: -2.863144
      10%: -2.567624
    → REJECT H0 (p=0.000000 < 0.05): Series is STATIONARY

--- 3b. KPSS Test ---
    H0: Series is STATIONARY (level or trend)
    Test statistic:     0.135425
    p-value:            0.100000
    Lags used:          9
    Critical values:
      10%: 0.347000
      5%: 0.463000
      2.5%: 0.574000
      1%: 0.739000
    → FAIL to reject H0 (p=0.100000 >= 0.05): Consistent with STATIONARITY

--- Combined Stationarity Verdict ---
  ADF  (H0=non-stationary): STATIONARY ✓
  KPSS (H0=stationary):     STATIONARY ✗
  ✓ Agreement: Series is STATIONARY (ADF rejects unit root, KPSS fails to reject stationarity)


C:\Users\Himaska\AppData\Local\Temp\ipykernel_22160\1393748828.py:33: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kpss_result = kpss(durations, regression='c', nlags='auto')


# Heteroskedasticity Test
Using rolling variance and ARCH LM test to identify for conditional variance.

In [3]:
# ═══════════════════════════════════════════════════════════════════════════
#   SECTION 4 — HETEROSKEDASTICITY TESTS
# ═══════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 72)
print("  SECTION 4 — HETEROSKEDASTICITY TESTS")
print("=" * 72)

# ARIMA assumes constant variance (homoskedasticity) of residuals.
# If variance changes over time, ARIMA's assumptions are violated.

# --- 4a. Rolling Variance Visualization Check ---
rolling_var = pd.Series(durations).rolling(50).var().dropna().values
print("\n--- 4a. Rolling Variance (50-window) ---")
print(f"    Rolling variance range: [{rolling_var.min():.2f}, {rolling_var.max():.2f}]")
print(f"    Rolling variance mean:  {rolling_var.mean():.2f}")
print(f"    Max/Min ratio:          {rolling_var.max() / max(rolling_var.min(), 0.001):.2f}x")

# --- 4b. ARCH LM Test ---
# H0: No ARCH effects (homoskedasticity)
# H1: ARCH effects present (heteroskedasticity / volatility clustering)
print("\n--- 4b. ARCH LM Test (Engle, 1982) ---")
print("    H0: No ARCH effects → HOMOSKEDASTICITY")
arch_result = het_arch(durations, nlags=10)
arch_stat, arch_pval, arch_fstat, arch_fpval = arch_result

print(f"    LM statistic:      {arch_stat:.4f}")
print(f"    p-value:           {arch_pval:.6f}")
print(f"    F-statistic:       {arch_fstat:.4f}")
print(f"    F p-value:         {arch_fpval:.6f}")

if arch_pval < 0.05:
    print(f"    → REJECT H0 (p={arch_pval:.6f} < 0.05): ARCH EFFECTS DETECTED — heteroskedasticity ✓")
    print("      → Consider a GARCH-family extension or variance-stabilizing transformation.")
else:
    print(f"    → FAIL to reject H0 (p={arch_pval:.6f} >= 0.05): No significant ARCH effects detected")


  SECTION 4 — HETEROSKEDASTICITY TESTS

--- 4a. Rolling Variance (50-window) ---
    Rolling variance range: [10.27, 58.74]
    Rolling variance mean:  28.85
    Max/Min ratio:          5.72x

--- 4b. ARCH LM Test (Engle, 1982) ---
    H0: No ARCH effects → HOMOSKEDASTICITY
    LM statistic:      85.2040
    p-value:           0.000000
    F-statistic:       8.8895
    F p-value:         0.000000
    → REJECT H0 (p=0.000000 < 0.05): ARCH EFFECTS DETECTED — heteroskedasticity ✓
      → Consider a GARCH-family extension or variance-stabilizing transformation.


# Autocorrelation
ARIMA requires model to be stationary and autocorrelated for it to predict.

In [4]:
# ═══════════════════════════════════════════════════════════════════════════
#   SECTION 5 — AUTOCORRELATION TESTS
# ═══════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 72)
print("  SECTION 5 — AUTOCORRELATION TESTS")
print("=" * 72)

# --- 5a. Ljung-Box Test (Multi-lag Portmanteau Test) ---
# H0: No autocorrelation up to lag k
# H1: At least one autocorrelation is non-zero
print("\n--- 5a. Ljung-Box Test ---")
print("    H0: No autocorrelation up to lag k")
lb_results = acorr_ljungbox(durations, lags=[5, 10, 20, 30], return_df=True)
print(lb_results.to_string())

has_autocorr = (lb_results['lb_pvalue'] < 0.05).any()
if has_autocorr:
    print("\n    → REJECT H0: Significant autocorrelation detected across lags.")
else:
    print("\n    → FAIL to reject H0: No significant autocorrelation detected (behaves like white noise).")

# --- 5b. ACF/PACF Analysis ---
acf_vals, acf_ci = acf(durations, nlags=20, alpha=0.05)
pacf_vals, pacf_ci = pacf(durations, nlags=20, alpha=0.05)

print("\n--- 5b. ACF/PACF Structure (First 20 Lags) ---")
sig_acf_lags = [i for i in range(1, 21) if abs(acf_vals[i]) > (acf_ci[i, 1] - acf_vals[i])]
sig_pacf_lags = [i for i in range(1, 21) if abs(pacf_vals[i]) > (pacf_ci[i, 1] - pacf_vals[i])]

print(f"    Significant ACF lags (outside 95% CI): {len(sig_acf_lags)}/20 → {sig_acf_lags if sig_acf_lags else 'None'}")
print(f"    Significant PACF lags (outside 95% CI): {len(sig_pacf_lags)}/20 → {sig_pacf_lags if sig_pacf_lags else 'None'}")
print(f"    Expected significant by chance (alpha=0.05): ~1/20")

# --- Autocorrelation Structure Conclusion ---
print("\n--- Autocorrelation Structure Verdict ---")
if not has_autocorr and len(sig_acf_lags) <= 1 and len(sig_pacf_lags) <= 1:
    print("  ✓ Series behaves as WHITE NOISE (no structural autocorrelation)")
    print("    → ARIMA(0,0,0) or baseline model is appropriate.")
elif len(sig_acf_lags) > 0 or len(sig_pacf_lags) > 0:
    print("  ✓ Autocorrelation structure present — suitable for ARIMA modeling.")
    print("    → Use ACF/PACF significant lags to guide initial p and q parameter selection.")
else:
    print("  ⚠ Mixed or weak autocorrelation signals.")


  SECTION 5 — AUTOCORRELATION TESTS

--- 5a. Ljung-Box Test ---
    H0: No autocorrelation up to lag k
      lb_stat     lb_pvalue
5   59.060858  1.899913e-11
10  62.872356  1.032579e-09
20  71.287566  1.120929e-07
30  79.968176  1.996546e-06

    → REJECT H0: Significant autocorrelation detected across lags.

--- 5b. ACF/PACF Structure (First 20 Lags) ---
    Significant ACF lags (outside 95% CI): 3/20 → [1, 4, 5]
    Significant PACF lags (outside 95% CI): 6/20 → [1, 2, 3, 4, 5, 10]
    Expected significant by chance (alpha=0.05): ~1/20

--- Autocorrelation Structure Verdict ---
  ✓ Autocorrelation structure present — suitable for ARIMA modeling.
    → Use ACF/PACF significant lags to guide initial p and q parameter selection.


# Running ARIMA Model

In [5]:

# ═══════════════════════════════════════════════════════════════════════════
#  SECTION 7 — ARIMA MODEL FITTING & RESIDUAL DIAGNOSTICS
# ═══════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 72)
print("  SECTION 7 — ARIMA MODEL & RESIDUAL DIAGNOSTICS")
print("=" * 72)

# --- 7a. Grid Search for Best ARIMA ---
print("\n--- 7a. ARIMA Grid Search (p=0..4, d=0..1, q=0..3) ---")
y = durations_log  # log(1 + duration) as used in autocorrelation notebook
n = len(y)

best_aic, best_order = np.inf, None
best_bic = np.inf
results = []

for p, d, q in itertools.product(range(5), range(2), range(4)):
    try:
        fit = ARIMA(y, order=(p, d, q)).fit(method_kwargs={'disp': False})
        if fit.aic < best_aic:
            best_aic = fit.aic
            best_order = (p, d, q)
            best_bic = fit.bic
        results.append({'order': (p, d, q), 'aic': fit.aic, 'bic': fit.bic})
    except Exception:
        continue

print(f"    Best ARIMA order by AIC: {best_order} (AIC={best_aic:.2f}, BIC={best_bic:.2f})")
print(f"    Total models evaluated: {len(results)}")

# --- 7b. Fit Best Model ---
print(f"\n--- 7b. Fitting ARIMA{best_order} ---")
best_model = ARIMA(y, order=best_order).fit()
print(best_model.summary())

# --- 7c. Residual Diagnostics ---
residuals = best_model.resid

print(f"\n--- 7c. Residual Diagnostics ---")

# Ljung-Box on residuals (should be white noise if model is adequate)
print("  Ljung-Box on Residuals (H0: residuals are white noise):")
for lag in [5, 10, 20]:
    lb_res = acorr_ljungbox(residuals, lags=[lag], return_df=True)
    lb_stat_r = lb_res['lb_stat'].iloc[0]
    lb_pval_r = lb_res['lb_pvalue'].iloc[0]
    sig = "OK (white noise)" if lb_pval_r > 0.05 else "FAIL (structure remains)"
    print(f"    Lag {lag:2d}:  LB stat = {lb_stat_r:7.3f}  |  p-value = {lb_pval_r:.4f}  → {sig}")

# ARCH test on residuals
print("  ARCH LM on Residuals (H0: homoskedastic residuals):")
arch_res = het_arch(residuals, nlags=10)
arch_res_pval = arch_res[1]
if arch_res_pval < 0.05:
    print(f"    p = {arch_res_pval:.6f} → HETEROSKEDASTIC RESIDUALS remain")
    print(f"    → ARIMA failed to capture variance dynamics")
else:
    print(f"    p = {arch_res_pval:.6f} → Residuals are homoskedastic ✓")

# ACF of residuals (residuals should be white noise)
acf_res, acf_res_ci = acf(residuals, nlags=20, alpha=0.05)
sig_acf_res = sum(1 for i in range(1, 21) if abs(acf_res[i]) > (acf_res_ci[i, 1] - acf_res[i]))
print(f"    Significant residual ACF lags: {sig_acf_res}/20 (expect ~1 by chance)")

# Normality of residuals (JB test)
jb_res = jarque_bera(residuals)
print(f"    Jarque-Bera (residuals): stat={jb_res[0]:.3f}, p={jb_res[1]:.6f}")


  SECTION 7 — ARIMA MODEL & RESIDUAL DIAGNOSTICS

--- 7a. ARIMA Grid Search (p=0..4, d=0..1, q=0..3) ---


c:\Users\Himaska\Documents\Projects\Business_Projects\intraday-momentum-composite\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\Himaska\Documents\Projects\Business_Projects\intraday-momentum-composite\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\Himaska\Documents\Projects\Business_Projects\intraday-momentum-composite\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


    Best ARIMA order by AIC: (1, 0, 1) (AIC=4108.29, BIC=4130.29)
    Total models evaluated: 40

--- 7b. Fitting ARIMA(1, 0, 1) ---
                               SARIMAX Results                                
Dep. Variable:                      y   No. Observations:                 1808
Model:                 ARIMA(1, 0, 1)   Log Likelihood               -2050.147
Date:                Tue, 21 Jul 2026   AIC                           4108.294
Time:                        12:45:44   BIC                           4130.294
Sample:                             0   HQIC                          4116.414
                               - 1808                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          1.5754      0.021     76.161      0.000       1.535       1.61

# Using ARIMA to solve for Autocorrelation
The best model used to model the RSI streak is ARIMA.
- Model could still fail as ARIMA assumes the autocorrelation and variance of RSI streak is a constant.

In [6]:

# ═══════════════════════════════════════════════════════════════════════════
#  SECTION 8 — WALK-FORWARD VALIDATION
# ═══════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 72)
print("  SECTION 8 — WALK-FORWARD OUT-OF-SAMPLE VALIDATION")
print("=" * 72)

min_train = int(n * 0.5)
refit_every = 5

preds_log, actual_log = [], []
fitted_model = None
order = best_order

for i in range(min_train, n):
    if fitted_model is None or (i - min_train) % refit_every == 0:
        fitted_model = ARIMA(y[:i], order=order).fit(method_kwargs={'disp': False})
    else:
        fitted_model = fitted_model.append([y[i-1]], refit=False)
    preds_log.append(fitted_model.forecast(steps=1)[0])
    actual_log.append(y[i])

preds = np.expm1(np.array(preds_log))
actual = np.expm1(np.array(actual_log))

from sklearn.metrics import mean_absolute_error, mean_squared_error

mae_arima = mean_absolute_error(actual, preds)
rmse_arima = np.sqrt(mean_squared_error(actual, preds))

# Benchmarks
naive_prev = np.expm1(y[min_train-1:n-1])
naive_mean = np.full_like(actual, np.expm1(y[:min_train].mean()))

print(f"\n    Walk-forward test samples: {len(actual)}")
print(f"    ARIMA{best_order}       MAE: {mae_arima:.2f} bars | RMSE: {rmse_arima:.2f}")
print(f"    Naive(prev)       MAE: {mean_absolute_error(actual, naive_prev):.2f} bars")
print(f"    Naive(mean)       MAE: {mean_absolute_error(actual, naive_mean):.2f} bars")

# Directional accuracy
direction_actual = actual > naive_prev
direction_pred = preds > naive_prev
dir_acc = (direction_actual == direction_pred).mean()
print(f"    Directional accuracy: {dir_acc:.1%}  (50% = coin flip)")

# Binomial test for directional accuracy vs 50%
from scipy.stats import binomtest
n_test = len(preds)
n_correct = int((direction_actual == direction_pred).sum())
binom_pval = binomtest(n_correct, n_test, 0.5, alternative='greater').pvalue
print(f"    Binomial test p-value (vs 50%): {binom_pval:.4f}")
if binom_pval < 0.05:
    print(f"    → Directional accuracy is STATISTICALLY SIGNIFICANT (p={binom_pval:.4f})")
else:
    print(f"    → Directional accuracy is NOT significant (p={binom_pval:.4f}) — at chance level")



  SECTION 8 — WALK-FORWARD OUT-OF-SAMPLE VALIDATION

    Walk-forward test samples: 904
    ARIMA(1, 0, 1)       MAE: 3.79 bars | RMSE: 5.28
    Naive(prev)       MAE: 4.85 bars
    Naive(mean)       MAE: 3.81 bars
    Directional accuracy: 65.7%  (50% = coin flip)
    Binomial test p-value (vs 50%): 0.0000
    → Directional accuracy is STATISTICALLY SIGNIFICANT (p=0.0000)


In [ ]:
# Generate the ACF/PACF plots
print("\nGenerating diagnostic plots...")
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Raw series
axes[0, 0].plot(durations, color='steelblue', alpha=0.7)
axes[0, 0].axhline(durations.mean(), color='red', linestyle='--', alpha=0.5, label=f'Mean={durations.mean():.1f}')
axes[0, 0].set_title('Streak Duration Time Series', fontsize=11)
axes[0, 0].set_xlabel('Streak #')
axes[0, 0].set_ylabel('Duration (bars)')
axes[0, 0].legend(fontsize=8)
axes[0, 0].grid(True, alpha=0.3)

# Rolling variance
axes[0, 1].plot(rolling_var, color='darkorange', alpha=0.7)
axes[0, 1].axhline(rolling_var.mean(), color='red', linestyle='--', alpha=0.5, label=f'Mean Var={rolling_var.mean():.2f}')
axes[0, 1].set_title('Rolling Variance (window=50)', fontsize=11)
axes[0, 1].set_xlabel('Streak #')
axes[0, 1].set_ylabel('Variance')
axes[0, 1].legend(fontsize=8)
axes[0, 1].grid(True, alpha=0.3)

# Distribution
axes[0, 2].hist(durations, bins=50, density=True, alpha=0.6, color='steelblue', edgecolor='white')
x_range = np.linspace(durations.min(), durations.max(), 100)
axes[0, 2].plot(x_range, stats.expon(*stats.expon.fit(durations)).pdf(x_range), 
                'r-', alpha=0.5, label='Exponential fit')
axes[0, 2].set_title(f'Distribution (n={len(durations)})', fontsize=11)
axes[0, 2].set_xlabel('Duration (bars)')
axes[0, 2].set_ylabel('Density')
axes[0, 2].legend(fontsize=8)
axes[0, 2].grid(True, alpha=0.3)

# ACF
plot_acf(durations, lags=40, ax=axes[1, 0], alpha=0.05)
axes[1, 0].set_title('ACF of Streak Durations', fontsize=11)

# PACF
plot_pacf(durations, lags=40, ax=axes[1, 1], alpha=0.05)
axes[1, 1].set_title('PACF of Streak Durations', fontsize=11)

# Residuals ACF
if best_model is not None:
    plot_acf(residuals, lags=40, ax=axes[1, 2], alpha=0.05)
    axes[1, 2].set_title(f'ACF of ARIMA{best_order} Residuals', fontsize=11)

plt.suptitle('ARIMA Hypothesis Testing — Diagnostic Suite', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.plot()
print("\nDone.")


Generating diagnostic plots...

Done.
